### Concept 1: What polymorphism means

#### the same method name, called the same way, behaving differently depending on which object's class it actually belongs to

In [ ]:
class Cat:
    def speak(self):
        return "Meow"

class Dog:
    def speak(self):
        return "Woof"

class Cow:
    def speak(self):
        return "Moo"

animals = [Cat(), Dog(), Cow()]

for animal in animals:
    print(animal)
    print(type(animal).__name__, "says:", animal.speak())

### Concept 2: Polymorphism via inheritance

In [ ]:
# for demonstration need to add init to it
class Account:
    def account_type(self):
        return "Generic Account"

class SavingsAccount(Account):
    def account_type(self):
        return "Savings Account"

class CheckingAccount(Account):
    def account_type(self):
        return "Checking Account"

accounts = [
    Account("Rahul", 500),
    SavingsAccount("Kavya", 1000),
    CheckingAccount("Priya", 750),
]

for acc in accounts:
    print(acc.owner, "->", acc.account_type())

### Concept 3: Duck typing

In [ ]:
class Duck:
    def quack(self):
        return "Quack!"

class Person:
    def quack(self):
        return "I'm pretending to be a duck!"

def make_it_quack(thing):
    print(thing.quack())    # never checks what 'thing' actually is

make_it_quack(Duck())
make_it_quack(Person())

Duck and Person share zero relationship — no common parent, no declared interface, nothing connecting them at all. make_it_quack never asks "are you a Duck?" anywhere — it just tries thing.quack() and trusts it'll work. It did, for both.
This is actually the exact same thing that happened back in Concept 1 with Cat/Dog/Cow — those three had no shared parent either, and the loop still worked identically for all of them. That wasn't a special case — it's just how Python always operates.

In [ ]:
class Cat:
    def meow(self):
        return "Meow!"

make_it_quack(Cat())

Python never checks upfront whether thing is "the right type" before running make_it_quack. It only discovers the problem at the exact moment it tries thing.quack() and that method genuinely isn't there. There's no gate checked in advance — just an attempt, which either finds the method and works, or doesn't and crashes right there.

#### if it walks like a duck and quacks like a duck, treat it as a duck" — Python doesn't care what class an object officially belongs to; it only cares whether the object actually has the method being called, at the moment it's called.

### Concept 4: Why duck typing works in Python

In [ ]:
def make_it_quack(thing: Duck) -> None:   # type hint claims 'thing' should be a Duck...
    print(thing.quack())

make_it_quack(Person())   # ...but Python runs this anyway, no check performed at all

I wrote thing: Duck — a type hint explicitly claiming this function only accepts a Duck. I then passed it a Person instead. Nothing stopped it. It ran completely fine. That's the key fact: in Python, type hints like thing: Duck are purely informational — decoration for humans and external tools (like mypy), never actually checked or enforced while the program runs.

 There's no gatekeeper checking types ahead of time — Python just runs your code, and the very first time it tries thing.quack(), it looks at whatever object is actually sitting there right now and asks "does this specific thing have a quack method?" That's it. No upfront verification, no advance promise required — just a direct, in-the-moment attempt. That absence of an advance type-checking gate is precisely what allows duck typing to exist at all.

### Concept 5: isinstance() checks vs. trusting duck typing

In [ ]:
def make_it_quack_bad(thing):
    if isinstance(thing, Duck):
        print(thing.quack())
    elif isinstance(thing, Person):
        print(thing.quack())
    else:
        raise TypeError("must be a Duck or Person")

def make_it_quack_good(thing):
    print(thing.quack())

make_it_quack_good(Robot())    # a brand new class, never mentioned anywhere
make_it_quack_bad(Robot())

#### if deleting the isinstance check would make previously-broken code start working fine — like Robot — the check was pointless, remove it, trust duck typing. If deleting the check still leaves you with an error either way — like "fifty" — the check isn't blocking anything that would've worked; it's just making the same, inevitable failure clearer and catching it sooner. That's the real difference: one check restricts things that were never actually broken, the other just improves the failure message for things that were always going to fail

### Duck typing 

In [7]:
class OpenRouterProvider:
    def generate(self, prompt):
        return f"[OpenRouter] response to: {prompt}"

class AnthropicProvider:
    def generate(self, prompt):
        return f"[Anthropic] response to: {prompt}"

def route_request(provider, prompt):
    return provider.generate(prompt)    # NO isinstance check anywhere

print(route_request(OpenRouterProvider(), "hello"))
print(route_request(AnthropicProvider(), "hello"))

class OllamaProvider:                    # added months later
    def generate(self, prompt):
        return f"[Ollama] response to: {prompt}"

print(route_request(OllamaProvider(), "hello"))

[OpenRouter] response to: hello
[Anthropic] response to: hello
[Ollama] response to: hello


#### route_request never checks which provider it got. When you add OllamaProvider months later, you don't touch route_request at all — it already works, because it never cared about which provider it was given, only that it has .generate(). This is the actual, practical reason your router can support new providers without rewriting the routing logic every time.

### isinstance — validating the prompt itself

In [ ]:
def classify_prompt_complexity(prompt):
    if not isinstance(prompt, str):
        raise TypeError(f"prompt must be a string, got {type(prompt).__name__}")
    return "simple" if len(prompt) < 50 else "complex"

print(classify_prompt_complexity("What is 2+2?"))
classify_prompt_complexity(12345)   # someone accidentally passed a number

Here you're not asking "which provider is this" — you're validating that the actual data (the prompt) is usable at all before you try to measure its length or feed it to a model. If you skipped this check, passing 12345 by mistake would fail later with object of type 'int' has no len() — confusing, and possibly buried deep inside your classifier logic instead of caught right at the door.
So, concretely, for your project: route_request should stay duck-typed — never check which provider class you got. Anything that validates the actual prompt, config values, or API responses coming in from outside your system — those are exactly where isinstance earns its place.

### Concept 6: BPE vs. WordPiece tokenizers

In [9]:
class BPETokenizer:
    def encode(self, text):
        return [ord(c) for c in text]                # character-level, BPE-style

    def decode(self, tokens):
        return "".join(chr(t) for t in tokens)

class WordPieceTokenizer:
    def encode(self, text):
        return [hash(word) % 1000 for word in text.split()]   # word-level, WordPiece-style

    def decode(self, tokens):
        return f"[decoded {len(tokens)} wordpiece tokens]"

def run_pipeline(tokenizer, text):
    tokens = tokenizer.encode(text)     # never checks which tokenizer this is
    print(f"{type(tokenizer).__name__} encoded '{text}' -> {tokens}")
    return tokenizer.decode(tokens)

print(run_pipeline(BPETokenizer(), "Hi"))
print(run_pipeline(WordPieceTokenizer(), "Hi there friend"))

BPETokenizer encoded 'Hi' -> [72, 105]
Hi
WordPieceTokenizer encoded 'Hi there friend' -> [772, 27, 343]
[decoded 3 wordpiece tokens]
